# Pytorch + Convultional

## 1. Datasets & DataLoaders (MNIST)
Load the data

In [57]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

## 2. Build Model
Create a simple NeuronalNetwork using Pytorch and adding the "Conv2d" and "MaxPool2d" layers.

In [76]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),            
            nn.Linear(1600, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NeuralNetwork().to(device)

## 3. Optimization
Train the model

In [77]:

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [78]:
learning_rate = 0.1
batch_size = 64
epochs = 10

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.293030  [   64/60000]
loss: 0.714965  [ 6464/60000]
loss: 0.445673  [12864/60000]
loss: 0.621804  [19264/60000]
loss: 0.584411  [25664/60000]
loss: 0.547893  [32064/60000]
loss: 0.425432  [38464/60000]
loss: 0.536013  [44864/60000]
loss: 0.597999  [51264/60000]
loss: 0.391928  [57664/60000]
Test Error: 
 Accuracy: 82.1%, Avg loss: 0.467208 

Epoch 2
-------------------------------
loss: 0.343442  [   64/60000]
loss: 0.422632  [ 6464/60000]
loss: 0.287416  [12864/60000]
loss: 0.426874  [19264/60000]
loss: 0.425780  [25664/60000]
loss: 0.427784  [32064/60000]
loss: 0.311375  [38464/60000]
loss: 0.537354  [44864/60000]
loss: 0.567449  [51264/60000]
loss: 0.365198  [57664/60000]
Test Error: 
 Accuracy: 85.9%, Avg loss: 0.379259 

Epoch 3
-------------------------------
loss: 0.257282  [   64/60000]
loss: 0.339301  [ 6464/60000]
loss: 0.240570  [12864/60000]
loss: 0.360075  [19264/60000]
loss: 0.358744  [25664/60000]
loss: 0.413030  [32064/600

## 4. Save and load
Save and load the model

In [74]:
import torchvision.models as models
model = models.vgg16(weights='IMAGENET1K_V1')
torch.save(model.state_dict(), 'model_weights.pth')

In [75]:
model = torch.load('model_weights.pth', weights_only=False)
print("Model loaded")

Model loaded
